# Function Setup

## 5.) Clustering related methods

In [43]:
# Get the number of members for each cluster
def kmeans_counts(data_labels, clusters =100):
    
    counts = np.zeros(shape = (clusters,1))
    for i in range(clusters):
        counts[i] = (data_labels == i).sum()
    
    return counts

## Method 1: Use labeling of k-Means Clustering and optimize the corresponding representative

In [178]:
## This Procedure combines the functions 'get_representatives_model', 'get_representative_configuration' and
# 'get_representatives_train', which operate all on a single model basis
# Now we combine the procedures for the list of models for all clusters
# Automize Procedure to obtain representatives of pre-determined clusters
# Input: List of data for given number of clusters.
# Optional input: loss_type, patience of early stopping (es_patience), model name (for saving the best model), 
#                 epochs
#                optimizer, learning rate of optimizer, decay of optimizer
# Output: list of representatives and list of models

def cluster_ann(y_lst, model_prediction, N_ensembles = 5, N_features = 5, scale = 1, N_input = 1,
                optimizer = 'adam', loss_type = 'mse', metric_type = 'mae',
                init_centroids = None, option_centroid = True, 
                input_option_cluster_scale = False, cluster_member_count = [None],
                N_epochs = 4000, es_patience = 15, 
                option_es = True, option_log_scaling = True, wd_cluster= None):

    t_start = time.time()
    # Parameters
    N_clusters = len(y_lst)
    N_output = y_lst[0].shape[1]
    representatives = np.zeros(shape=(N_clusters,N_features))
    representatives_pv = np.zeros(shape=(N_clusters,N_output))
    history = []
    times = np.zeros(N_clusters)


    # Get general model (for all clusters)
    # Option 1: Number of members of cluster not considered
    if input_option_cluster_scale == False:
        model_clustering = get_representatives_model(N_input=1,  
                                                     N_features= N_features,
                                                     scale=scale, N_ensembles=N_ensembles, 
                                                     N_repeat = N_output, N_output= N_output,
                                                     log_scaling= option_log_scaling)
    else:
        # Option 2: We include the number of members of the cluster to fine-tune training
        model_clustering = get_representatives_model(N_input=1,  input_option_cluster_scale= True,
                                                     N_features= N_features,
                                                     scale=scale, N_ensembles=N_ensembles, 
                                                     N_repeat = N_output, N_output= N_output,
                                                     log_scaling= option_log_scaling)

    
    if len(init_centroids) != N_clusters:
        #Get initial weights, to reset them for every cluster (otherwise use K-means centroids as init-values)
        w_clustering_init = [model_clustering.layers[1].get_weights()]

    
    # Import weights for model (Note: Including a constraint influences the number of layers)
    get_representative_configuration(model_update = model_clustering, model_weights= model_prediction, 
                                         layer_start = 2)  
    
    
    if N_input != 1: # For N_input == 1 we simply retrieve the weights (no bias included)
        # Function to obtain representative contracts (i.e. hidden output)
        get_cluster_representative = K.function(inputs = [model_clustering.layers[0].input],
                                      outputs=[model_clustering.layers[1].output])
    
    print('Model set up. Time required: ' + str(np.round_(time.time()-t_start, decimals = 2))+ ' sec.')
    
    for i in range(N_clusters):
        #if i>0:
        # reset (trainable) weights
        if len(init_centroids) != N_clusters:
            model_clustering.layers[1].set_weights(w_clustering_init)
        else:
            # Note: a^(1) should be set to equal the centroids and there is still a tanh activation
            # Hence, use arctanh transform for weights; There is no bias parameter.
            model_clustering.layers[1].set_weights([np.arctanh(init_centroids[i,:]).reshape((1,N_features))])
        

        t_start = time.time()
        print('Model for Cluster {} of {}'.format(i+1,len(y_lst)))        
        print('\t Training in progress')
        
        
        # Train model (e.g. mit AdaDelta optimizer)
        # Model Option 1
        if input_option_cluster_scale == False:
            hist = get_representatives_train(model = model_clustering, y = y_lst[i], 
                                             N_epochs = N_epochs, 
                                             optimizer= optimizer, loss = loss_type, metric = metric_type,
                                              es_patience = es_patience, cluster_number = str(i), 
                                              option_es = option_es, wd_cluster=wd_cluster)
            
        else:
            # Model Option 2
            hist = get_representatives_train(model = model_clustering, y = y_lst[i]*cluster_member_count[i], 
                                              N_epochs = N_epochs,
                                              optimizer= optimizer, loss = loss_type, metric = metric_type,
                                              es_patience = es_patience, cluster_number = str(i), 
                                              option_cluster_scale= True, 
                                              cluster_member_count= cluster_member_count[i],
                                              option_es = option_es, wd_cluster=wd_cluster)
        
        
        history.append(hist.history)
        times[i] = np.round_(time.time()-t_start, decimals = 2)
        print(' \t Cluster {} completed. Time passed '.format(i+1)+ str(times[i])+ ' sec.')
        
        
        # Save representative (i.e. hidden output)
        if (N_input!=1) & (init_centroids.shape[0]!=N_clusters):
            representatives[i,:] = get_cluster_representative([np.array([1]*N_input).reshape((1,N_input))])[0]
        else:
            #### Attention !!!! Retrieving weights is much faster, than using the Keras function
            #### but the activation function has to be included to obtain representative contracts #########
            representatives[i,:] = np.tanh(model_clustering.layers[1].get_weights()[0])
        # Save corresponding reserve
        if input_option_cluster_scale == False:
            representatives_pv[i,:] = model_clustering.predict(x = np.array([1]*N_input).reshape((1,N_input)))[0]
        else:
            representatives_pv[i,:] = model_clustering.predict(x = [np.array([1]*N_input).reshape((1,N_input)),
                                                                    np.array([1], ndmin = 2)])[0]
            
    return [representatives, representatives_pv, model_weights, times, history]#, model_clustering]

In [177]:
# Create Model for the determination of a cluster's representative
# Procedure will have to be repeated for each cluster
# Input: N number of members of cluster, scale: scaling factor in Ensemble model
# Output: Model
# Options: mid_layer_option: include layer between flattened input and dense layer with 
#          _features units, to bundle information
#         contraint_option: Include unit_constraint (sum of exiting weights for all neuron <1) in all layers
#         qualitative_option: Include qualitative model

def get_representatives_model(N_input=None, input_option_cluster_scale = False, N_features = 5, 
                              scale=1, N_repeat = 41, N_output = 41, N_ensembles = 1, log_scaling = True,
                             option_adaptive_agglom = False, N_adaptive = 1):
    
    count = 0
    
    # Only the cummulative reserve of the cluster is relevant for the model, not the members' features
    # Model will find a representative contract to optimally fit the target (with no respect to the input contracts)
    INPUT = Input(shape = (N_input,))
    
    if input_option_cluster_scale:
        INPUT_scale = Input(shape = (1,))
    count +=1
    x = Dense(units = N_features, activation = 'tanh', use_bias = False)(INPUT)
    count +=1
        
    if N_ensembles >1:
        # include previous choice of Ensemble Model        
        OUTPUT = combine_models(input_layer=x, n_ensembles= N_ensembles, 
                                load_weights= False, weights_ensembles = None, 
                                scale = scale, LSTM_nodes= [n_output],  FFN_nodes = [n_output],
                                dense_act_fct= 'tanh',
                                return_option = 'output')
        
    else: # For a 1-Model-Ensemble the average-Layer cannot be evaluated
        
        # include previous choice of Ensemble Model        
        OUTPUT = create_rnn_model(model_input=x,widths_rnn= [N_output], widths_ffn=[N_output], 
                                optimizer_type='adam',loss_type='mse', 
                                metric_type='mae', dense_act_fct= 'tanh',
                                dropout_rnn=0, option_CUDA= True, option_recurrent_dropout= False,
                                lambda_layer = True, lambda_scale =scale, 
                                log_scale=log_scaling, return_option = 'output')
        
        
    if input_option_cluster_scale:
        OUTPUT = Lambda( lambda xvar: xvar*INPUT_scale)(OUTPUT)
        model = Model(inputs = [INPUT, INPUT_scale], outputs = OUTPUT)
    else:    
        model = Model(inputs = INPUT, outputs = OUTPUT)

    for i in range(count,len(model.layers)):
        model.layers[i].trainable = False
    
    return model

In [18]:
## This Procedure combines the functions 'get_representatives_model', 'get_representative_configuration' and
# 'get_representatives_train', which operate all on a single model basis
# Now we combine the procedures for the list of models for all clusters
# Automize Procedure to obtain representatives of pre-determined clusters
# Input: List of data for given number of clusters.
# Optional input: loss_type, patience of early stopping (es_patience), model name (for saving the best model), 
#                 epochs
#                optimizer, learning rate of optimizer, decay of optimizer
# Output: list of representatives and list of models

def cluster_ann_test(y_lst, model_prediction, N_ensembles = 5, N_features = 5, 
                     scale = 1, N_input = 1, N_centroids = 1,
                optimizer = 'adam', loss_type = 'mse', metric_type = 'mae',
                init_centroids = None, option_centroid = True, 
                cluster_member_count = [None],
                N_epochs = 4000, es_patience = 15, 
                option_es = True, option_log_scaling = True, wd_cluster= None):
    
    t_start = time.time()
    # Parameters
    N_clusters = len(y_lst)
    N_output = y_lst[0].shape[1]
    representatives = {} #np.zeros(shape=(N_clusters,N_features))
    representatives_pv = {} #np.zeros(shape=(N_clusters,N_output))
    history = {}#[]
    times = {}#np.zeros(N_clusters)
    
    
    # Get general model (for all clusters)
    # Option 1: Number of members of cluster not considered
    if N_centroids==1:
        model_clustering = get_representatives_model_test(N_input=1,  
                                                     N_features= N_features,
                                                     scale=scale, N_ensembles=N_ensembles, 
                                                     N_repeat = N_output, N_output= N_output,
                                                     log_scaling= option_log_scaling)
    else:
        # Option 2: We include the number of members of the cluster to fine-tune training
        model_clustering = get_representatives_model_test(N_input=1,  input_option_cluster_scale= True,
                                                     N_features= N_features,
                                                     scale=scale, N_ensembles=N_ensembles, 
                                                     N_repeat = N_output, N_output= N_output,
                                                     log_scaling= option_log_scaling, N_centroids= N_centroids)


        
        
        
        
#############################################################
#
#                Model Configuration 
#
##############################################################

    # Import weights for model (Note: Including a constraint influences the number of layers)
    get_representative_configuration(model_update = model_clustering, model_weights= model_prediction, 
                                     layer_start = 2, N_centroids= N_centroids, N_ensembles= N_ensembles)  

    
    print('Model set up. Time required: ' + str(np.round_(time.time()-t_start, decimals = 2))+ ' sec.')
    
##############################################################    
#
#       Training of Model (per Cluster) begins
#
##############################################################
    
    for i in range(N_clusters):
        
        #############################################################   
        ###################### Weight Initialization #################
        
        if (len(init_centroids) != N_clusters) or (option_centroid==False):
            # Use random, Sobol-generated inital weights for all centroids
            random = sobol_seq.i4_sobol_generate(dim_num = N_input, n = N_centroids)*2-1
            w_clustering_init = [random[k,:].reshape((1,N_features))for k in range(N_centroids)]


        else:
            # Use K-means centroids to boost performance
            # Mix different Ages to mix "distribution" of payout profile
            # For all features but Age use K-means centroid values
            
            if N_centroids>1:
                random = sobol_seq.i4_sobol_generate(dim_num = 1, n = int(N_centroids/2))
                w_low = [np.array([np.arctanh(init_centroids[i,0])]+
                                  list(init_centroids[i,1]+random[k]*(1-init_centroids[i,1]))+
                                  list(np.arctanh(init_centroids[i,2:]))
                                ).reshape((1,N_features)) for k in range(int(N_centroids/2))]
                w_up = [np.array([np.arctanh(init_centroids[i,0])]+
                                  list(init_centroids[i,1]-random[k]*(1+init_centroids[i,1]))+
                                  list(np.arctanh(init_centroids[i,2:]))
                                ).reshape((1,N_features)) for k in range(int(N_centroids/2))]
            
            
            if N_centroids ==1:
                w_clustering_init = [np.arctanh(init_centroids[i,:]).reshape((1,N_features))]
            elif N_centroids%2 == 0: 
                # Equal share of ANN centroids initialized left and right of K-means centroid
                w_clustering_init = w_low+w_up
            
            else:
                # N_centroids uneven -> Initialize 1x K-means centroid and distribute rest equally right and left
                w_clustering_init = [np.arctanh(init_centroids[i,:]).reshape((1,N_features))]+w_low+w_up
            
           # w_clustering_init = [np.array([np.arctanh(init_centroids[i,0])]+list(random[k])+
            #                                   list(np.arctanh(init_centroids[i,2:]))
             #                            ).reshape((1,N_features)) for k in range(N_centroids)]
            
            
            
            
        #############################################################   
        ######################     Set Weights      #################
        
        
        
        for k in range(N_centroids):
            model_clustering.get_layer(name='Centroid_{}'.format(k)).set_weights([w_clustering_init[k]])
                

        
        ##################################################################################

        t_start = time.time()
        print('Model for Cluster {} of {}'.format(i+1,len(y_lst)))        
        print('\t Training in progress')
        
        
        # Train model (e.g. mit AdaDelta optimizer)
        # Model Option 1
        if N_centroids ==1:
            history[i] = get_representatives_train(model = model_clustering, y = y_lst[i], 
                                             N_epochs = N_epochs, 
                                             optimizer= optimizer, loss = loss_type, metric = metric_type,
                                              es_patience = es_patience, cluster_number = str(i), 
                                              option_es = option_es, wd_cluster=wd_cluster, count = i).history
            
        else:
            # Model Option 2
            history[i] = get_representatives_train(model = model_clustering, y = y_lst[i]*cluster_member_count[i], 
                                              N_epochs = N_epochs,
                                              optimizer= optimizer, loss = loss_type, metric = metric_type,
                                              es_patience = es_patience, cluster_number = str(i), 
                                              option_cluster_scale= True, 
                                              cluster_member_count= cluster_member_count[i],
                                              option_es = option_es, wd_cluster=wd_cluster, count = i).history
        
        
        times[i] = np.round_(time.time()-t_start, decimals = 2)
        print(' \t Cluster {} completed. Time passed '.format(i+1)+ str(times[i])+ ' sec.')
                
        # Save representative (i.e. hidden output)
        if (N_centroids > 1) or True :
            representatives[i] = [ np.tanh(model_clustering.get_layer(
                name = 'Centroid_{}'.format(k)).get_weights()[0]) for k in range(N_centroids)]
        else:
            #### Attention !!!! Retrieving weights is much faster, than using the Keras function
            #### but the activation function has to be included to obtain representative contracts #########
            representatives[i,:] = np.tanh(model_clustering.layers[1].get_weights()[0])
        # Save corresponding reserve
        #if input_option_cluster_scale == False:
         #   representatives_pv[i] = model_clustering.predict(x = np.array([1]*N_input).reshape((1,N_input)))[0]
        #else: 
         #   representatives_pv[i] = model_clustering.predict(x = [np.array([1]*N_input).reshape((1,N_input)),
          #                                                          np.array([cluster_member_count[i]], ndmin = 2)])[0]
        
        representatives_pv[i] = [ model_prediction.predict(x = [representatives[i][k]])[0] for k in range(N_centroids)]
        
        
    return [representatives, representatives_pv, times, history]#, model_clustering]

In [46]:
# Create Model for the determination of a cluster's representative
# Procedure will have to be repeated for each cluster
# Input: N number of members of cluster, scale: scaling factor in Ensemble model
# Output: Model
# Options: mid_layer_option: include layer between flattened input and dense layer with 
#          _features units, to bundle information
#         contraint_option: Include unit_constraint (sum of exiting weights for all neuron <1) in all layers
#         qualitative_option: Include qualitative model

def get_representatives_model_test(N_input=None, input_option_cluster_scale = False, N_features = 5, 
                              scale=1, N_repeat = 41, N_output = 41, N_ensembles = 1, log_scaling = True,
                             N_centroids = 1, option_adaptive_frequency = False):
        
    # Only the cummulative reserve of the cluster is relevant for the model, not the members' features
    # Model will find a representative contract to optimally fit the target (with no respect to the input contracts)
    
    #INPUT = {}
    OUTPUT = {}
    centroids_scale = {}
    x= {}
    
    # Input Layer: Overall Number of Contracts in Cluster -> for scaling
    if N_centroids>1:
        INPUT_scale = Input(shape = (1,), name = 'Input_Freq')
    
    # Contract detail related Input Layer
    INPUT = Input(shape = (N_input,))
        
    for i in range(N_centroids):
        
        x[i] = Dense(units = N_features, activation = 'tanh', use_bias = False, name = 'Centroid_'+str(i))(INPUT)
        
        # Layer for quantitative scaling, connected to overall number of contracts in cluster
        # Assign relative importances to centroids of clusters
        if N_centroids >1:
            centroids_scale[i] = Dense(units=1, activation = 'relu', use_bias = False, 
                                   name = 'Freq_'+str(i))(INPUT_scale)        
        
    if N_ensembles >1:
        
        for i in range(N_centroids):
            # include previous choice of Ensemble Model        
            OUTPUT[i] = combine_models(input_layer=x[i], n_ensembles= N_ensembles, 
                                    load_weights= False, weights_ensembles = None, 
                                    scale = scale, LSTM_nodes= [n_output], FFN_nodes = [n_output],
                                    dense_act_fct= 'tanh',
                                    return_option = 'output', index = str(i))
        
    else: # For a 1-Model-Ensemble the average-Layer cannot be evaluated

        for i in range(N_centroids):
            OUTPUT[i] = create_rnn_model(model_input=x[i],widths_rnn= [N_output], widths_ffn=[N_output], 
                                    optimizer_type='adam',loss_type='mse', 
                                    metric_type='mae', dense_act_fct= 'tanh',
                                    dropout_rnn=0, option_CUDA= True, option_recurrent_dropout= False,
                                    lambda_layer = True, lambda_scale =scale, 
                                    log_scale=log_scaling, return_option = 'output', branch_name = str(i))
        
    if input_option_cluster_scale & (N_centroids ==1):
        #OUTPUT[1] = Lambda( lambda xvar: xvar*INPUT_scale)(OUTPUT[1])
        OUTPUT[1] = multiply([OUTPUT[0],INPUT_scale])
        model = Model(inputs = [INPUT, INPUT_scale], outputs = OUTPUT[1])
    elif (input_option_cluster_scale==False) & (N_centroids ==1):
        model = Model(inputs = INPUT, outputs = [OUTPUT[0]])
    else: # N_centroids >0 (input_option_cluster_scale has to be set to True to have a reasonable model)
        OUTPUT_centroids = {}
        for k in range(N_centroids): 
            # Scale Policy Values/ Predictions of centroids to their importance (weighted number of occurence)
            OUTPUT_centroids[k] = multiply([OUTPUT[k],centroids_scale[k]])
            
        # Add weighted PV to obtain cluster-PV
        OUTPUT_final = add([OUTPUT_centroids[i] for i in range(N_centroids)])
        
        # create Model
        model = Model(inputs = [INPUT, INPUT_scale], outputs = [OUTPUT_final])
    
    
    # Set all layers after optimized centroid to non-trainable
    for i in range(len(model.layers)):
        model.layers[i].trainable = False
        
    for i in range(N_centroids):
        model.get_layer(name = 'Centroid_'+str(i)).trainable = True
        
    # Set frequencies of centroids trainable and set initial weight
    if (option_adaptive_frequency == True) or (N_centroids>1):
        for i in range(N_centroids):
            model.get_layer(name = 'Freq_'+str(i)).set_weights([np.array([1/N_centroids]).reshape((1,1))] )
            model.get_layer(name = 'Freq_'+str(i)).trainable = option_adaptive_frequency
        
    return model

In [183]:
# Transfer pretrained weights to model and set corresponding parts as non-trainable
# Input: weights
def get_representative_configuration(model_update, model_weights, layer_start = 2, N_ensembles = 1,
                                     N_centroids = 1,optimizer = 'adam', 
                                     loss = 'mse', metric = 'mae'):
    
    n = len(model_weights.layers)
    
    # Compile model
    # Note: Every time you compile, the weights are reset to default, random initialization
    model_update.compile(optimizer = optimizer, loss = loss, metrics = [metric] )
       
    # If model suitable, continue with layer-wise weights-transfer
    if N_centroids ==1:
        for i in range(n-1):
            # For model to update: skip 2 Layers (Input + Dense) to get to Ensemble Part
            # For Ensemble Model: skip 1 Layer (i.e. Input Layer)
            model_update.layers[layer_start+i].set_weights(model_weights.layers[1+i].get_weights())
    else:
        for i in range(N_centroids): # Only LSTM and Dense Layer need updated parameters
            for j in range(N_ensembles):
                if N_ensembles == 1: i=''
                model_update.get_layer(name='RNN_{}{}1'.format(i,j)).set_weights(
                        model_weights.get_layer(name='RNN_{}1'.format(j)).get_weights())
                model_update.get_layer(name='Dense_{}{}1'.format(i,j)).set_weights(
                        model_weights.get_layer(name='Dense_{}1'.format(j)).get_weights())
                
            #model_update.layers[1+layer_start*N_centroids+i].set_weights(model_weights.layers[2].get_weights())
            #model_update.layers[1+(layer_start+1)*N_centroids+i].set_weights(model_weights.layers[3].get_weights())
    return

In [51]:
## Train a single model of the list of clustering models
def get_representatives_train(model,y, N_epochs = 2000, 
                              optimizer = 'adam', loss = 'mse', metric = 'mae',
                              es_patience = 500, cluster_number = None, show_progress = False,
                              option_cluster_scale = False, cluster_member_count = None,
                              option_es = True, wd_cluster = None, count = 0,
                             option_import = True):
    
    model.compile(optimizer = optimizer, loss = loss, metrics = [metric])
    
    x = np.array([1])
    if option_cluster_scale:
        x = [x,cluster_member_count]
    
    if option_es == True:
        # patient early stopping; Note: In clustering there is no validation data
        es = EarlyStopping(monitor='loss', mode='min', verbose=show_progress, patience=es_patience)
        # save only best configuration
        mc = ModelCheckpoint(filepath = wd_cluster+r'\cluster_{}.h5'.format(count), monitor='loss', mode='min', 
                         verbose=0, save_best_only=True)
        hist = model.fit(x,y,batch_size=1, epochs = N_epochs, callbacks= [es, mc], verbose = show_progress)
    else:
        hist = model.fit(x,y,batch_size=1, epochs = N_epochs, verbose = show_progress) 
    
    if option_import&option_es:
        model.load_weights(wd_cluster+r'\cluster_{}.h5'.format(count))
    
    return hist

## Evaluate Results

In [1]:
## function which compares (or visualizes) agglomeration
## We want to compare the baseline kMeans with the ANN Agglomeration
## For the ANN we have to consider the predicted reserve (by part I), which
# is integrated in the ANN-Model for clustering
# and compare this to the reserve based on classical calculation and the representatives
# obtain by the ANN-Cluster-Model
# option_plot_selection: insert a a list of numbers, representing the selcted clusters to be plotted

def analyze_agglomeration(baseline, y, Max_min, x = None, include_ann = False,
                          ann_representatives = None, ann_prediction = None, 
                          option = 'plot', individual_clusters = False, option_cluster_fit = True,
                          n_columns = 5, figsize = (20,30), option_plot_selection = None, 
                          insurance_type = 'termlife', pension_age_max = 67,
                          interest_rate = 0.03, option_header = False, option_legend = True,
                          ep_rate = [0.3, 0.1, 0.1, 0.1, 0.1, 0.1]):
    
    #Number of clusters
    n_cl = baseline.cluster_centers_.shape[0]
    # Number of members per cluster
    count = kmeans_counts(baseline.labels_,n_cl) # customized function kmeans_counts
    # size of output (vector of policy values)
    n_out = targets.shape[1]
    
    # Retransformed Data
    if insurance_type == 'termlife':
        data_kmeans = data_re_transform_features(baseline.cluster_centers_, Max_min, option= 'conditional')
        if include_ann == True:
            data_ann = data_re_transform_features(ann_representatives, Max_min, option= 'conditional')
        
    elif insurance_type == 'pensions':
        data_kmeans = data_re_transform_features(baseline.cluster_centers_, Max_min, option= 'standard')
        if include_ann == True:
            data_ann = data_re_transform_features(ann_representatives, Max_min, option= 'standard')
    else:
        print('Error: insurance_type not known/ implemented!')
        return
   
    

    
    # Calculate targets for (floored and ceiled) centroids for baseline and
    # optionally: also for representatives obtained by ANN
    PV_km_low = np.zeros(shape = (n_cl, n_out))
    PV_km_up = np.zeros(shape = (n_cl, n_out))
    PV_ann_up = np.zeros(shape = (n_cl, n_out))
    PV_ann_low = np.zeros(shape = (n_cl, n_out))
    for i in range(n_cl):
    ##############################
        if insurance_type == 'termlife':
            PV_km_low[i,0:max(np.floor(data_kmeans[i,2]).astype('int')-
                                  np.floor(data_kmeans[i,3]).astype('int'),0)+1] = get_termlife_reserve_profile(
                                                        age_curr = np.floor(data_kmeans[i,0]).astype('int'), 
                                                        Sum_ins= data_kmeans[i,1],
                                                         duration = np.floor(data_kmeans[i,2]).astype('int'), 
                                                        #interest = data_kmeans[i,4],
                                                        interest = interest_rate,
                                                         age_of_contract = np.floor(data_kmeans[i,3]).astype('int'), 
                                                          option_past = False)

            PV_km_up[i,0:max(np.ceil(data_kmeans[i,2]).astype('int')-
                  np.ceil(data_kmeans[i,3]).astype('int'),0)+1] = get_termlife_reserve_profile(
                                                        age_curr = np.ceil(data_kmeans[i,0]).astype('int'), 
                                                        Sum_ins= data_kmeans[i,1],
                                                        duration = np.ceil(data_kmeans[i,2]).astype('int'), 
                                                        #interest = data_kmeans[i,4], 
                                                        interest = interest_rate,
                                                        age_of_contract = np.ceil(data_kmeans[i,3]).astype('int'), 
                                                        option_past = False)

            # Optional: Also for representatives of ANN
            if include_ann == True:
                PV_ann_low[i,0:max(np.floor(data_ann[i,2]).astype('int')-
                              np.floor(data_ann[i,3]).astype('int'),0)+1] = get_termlife_reserve_profile(
                                                            age_curr = np.floor(data_ann[i,0]).astype('int'), 
                                                            Sum_ins= data_ann[i,1],
                                                            duration = np.floor(data_ann[i,2]).astype('int'), 
                                                            #interest = data_ann[i,4], 
                                                            interest = interest_rate,
                                                            age_of_contract = np.floor(data_ann[i,3]).astype('int'), 
                                                            option_past = False)

                PV_ann_up[i,0:max(np.ceil(data_ann[i,2]).astype('int')-
                              np.ceil(data_ann[i,3]).astype('int'),0)+1] = get_termlife_reserve_profile(
                                                            age_curr = np.ceil(data_ann[i,0]).astype('int'), 
                                                            Sum_ins= data_ann[i,1],
                                                            duration = np.ceil(data_ann[i,2]).astype('int'), 
                                                            #interest = data_ann[i,4], 
                                                            interest = interest_rate,
                                                            age_of_contract = np.ceil(data_ann[i,3]).astype('int'), 
                                                            option_past = False)
                
    ############################
        elif insurance_type == 'pensions':
            # input_used = ['Fund','Age', 'Salary', 'Salary_scale', 'Contribution', 'interest_rate']
            PV_km_low[i,0:max(pension_age_max- np.floor(data_kmeans[i,1]).astype('int'),0)+1] =  get_pension_reserve(fund_accum = data_kmeans[i,0], 
                                     age = np.floor(data_kmeans[i,1]).astype('int'), 
                                     salary = data_kmeans[i,2], salary_scale = data_kmeans[i,3], 
                                     contribution = data_kmeans[i,4], 
                                     A = 0.00022, B = 2.7*10**(-6), c = 1.124, 
                                    #interest = data_kmeans[i,5],
                                     interest = interest_rate,
                                     pension_age_max = pension_age_max, 
                                     early_pension = ep_rate)

            #print(PV_km_low[i,0:max(pension_age_max- np.floor(data_kmeans[i,1]).astype('int'),0)+1])
            #print(data_kmeans[i,:])




            PV_km_up[i,0:max(pension_age_max- np.ceil(data_kmeans[i,1]).astype('int'),0)+1] = get_pension_reserve(fund_accum =data_kmeans[i,0], 
                                     age = np.ceil(data_kmeans[i,1]).astype('int'), 
                                     salary = data_kmeans[i,2], salary_scale = data_kmeans[i,3], 
                                     contribution = data_kmeans[i,4], 
                                     A = 0.00022, B = 2.7*10**(-6), c = 1.124, 
                                      #interest = data_kmeans[i,5],
                                      interest = interest_rate,
                                     pension_age_max = pension_age_max, 
                                     early_pension = ep_rate)

            # Optional: Also for representatives of ANN
            if include_ann == True:
                PV_ann_low[i,0:max(pension_age_max- np.floor(data_ann[i,1]).astype('int'),0)+1] = get_pension_reserve(fund_accum =data_ann[i,0], 
                                         age = np.floor(data_ann[i,1]).astype('int'), 
                                         salary = data_ann[i,2], salary_scale = data_ann[i,3], 
                                         contribution = data_ann[i,4], 
                                         A = 0.00022, B = 2.7*10**(-6), c = 1.124, 
                                          #interest = data_ann[i,5],
                                          interest = interest_rate,
                                         pension_age_max = pension_age_max, 
                                         early_pension = ep_rate)

                PV_ann_up[i,0:max(pension_age_max- np.ceil(data_ann[i,1]).astype('int'),0)+1] = get_pension_reserve(fund_accum =data_ann[i,0], 
                                         age = np.ceil(data_ann[i,1]).astype('int'), 
                                         salary = data_ann[i,2], salary_scale = data_ann[i,3], 
                                         contribution = data_ann[i,4], 
                                         A = 0.00022, B = 2.7*10**(-6), c = 1.124, 
                                        #interest = data_ann[i,5],
                                        interest = interest_rate,
                                         pension_age_max = pension_age_max, 
                                         early_pension = ep_rate)



        else:
            print('Error: insurance_type not known/ implemented!')
            return

    ##############################################
    # Calculate actual targets per cluster 
    targets_cl = np.zeros(shape = (n_cl, n_output))
    for i in range(n_cl):
        index = baseline.labels_ == i
        targets_cl[i,:] = targets[index,:].sum(axis=0)/count[i]

 
    
    if option == 'plot':
        
        if individual_clusters == True:
            
            
            if option_plot_selection == None: # Plot all clusters C_1,..,C_K
                fig, ax = plt.subplots(nrows = np.ceil(n_cl/n_columns).astype('int'), 
                                       ncols = n_columns, figsize = figsize)
                ax = ax.flatten()

                for i in range(n_cl):
                    # Actual Targets
                    ax[i].plot(targets_cl[i,:], 'r*', label = 'Target')
                    # Reserve based on K-Means clustering
                    ax[i].plot(PV_km_up[i,:], linestyle = ':', color = 'orange', 
                               label = 'K-means Bounds')
                    ax[i].plot(PV_km_low[i,:], linestyle = ':', color = 'orange')
                    ax[i].plot((PV_km_up[i,:]+PV_km_low[i,:])/2, color = 'orange', 
                               label = 'K-means Prediction')

                    if i%n_columns==0: # first column
                        ax[i].set_ylabel('Policy Value', fontsize = 'large')
                    if i>= (n_columns*(np.ceil(n_cl/n_columns).astype('int')-1)): # last row
                        ax[i].set_xlabel('Time, $t$', fontsize = 'large')

                    if include_ann == True:
                        # Predicted Reserve by ANN
                        ax[i].plot(ann_prediction[i,:], color = 'black', label = 'ANN Prediction')
                        # Reserve based on classical calculation using representative contracts of ANN approach
                        ax[i].plot(PV_ann_up[i,:], color = 'black',linestyle = ':', 
                                   label = 'ANN Bounds')
                        ax[i].plot(PV_ann_low[i,:],  color = 'black',linestyle = ':') 

                    if i == 0:
                        ax[i].legend()

                plt.tight_layout()
            else: ### Plot only selection of clusters C_1,...,C_[option_plot_selection]
                fig, ax = plt.subplots(nrows = np.ceil(len(option_plot_selection)/n_columns).astype('int'), 
                                       ncols = n_columns, figsize = figsize)
                ax = ax.flatten()

                for i in range(len(option_plot_selection)):
                    # Actual Targets
                    ax[i].plot(targets_cl[option_plot_selection[i],:], 'r*', label = 'Target')
                    # Reserve based on K-Means clustering
                    ax[i].plot(PV_km_up[option_plot_selection[i],:], linestyle = ':', color = 'orange', 
                               label = 'K-means Bounds')
                    ax[i].plot(PV_km_low[option_plot_selection[i],:], linestyle = ':', color = 'orange')
                    ax[i].plot((PV_km_up[option_plot_selection[i],:]+PV_km_low[option_plot_selection[i],:])/2, color = 'orange', 
                               label = 'K-means Prediction')
                    
                    if i%n_columns==0: # first column
                        ax[i].set_ylabel('Policy Value', fontsize = 'large')
                    if i>= (len(option_plot_selection)-n_columns): # last row
                        ax[i].set_xlabel('Time, $t$', fontsize = 'large')

                    if include_ann == True:
                        # Predicted Reserve by ANN
                        ax[i].plot(ann_prediction[option_plot_selection[i],:], color = 'black', 
                                   label = 'ANN Prediction')
                        # Reserve based on classical calculation using representative contracts of ANN approach
                        ax[i].plot(PV_ann_up[option_plot_selection[i],:], color = 'black', linestyle = ':', 
                                   label = 'ANN Bounds')
                        ax[i].plot(PV_ann_low[option_plot_selection[i],:], color = 'black', linestyle = ':') 

                    if i == 0:
                        ax[i].legend()
                plt.tight_layout()
            
        
        ### Look at cumulative cluster fit ###
        fig_cum, ax_cum = plt.subplots(1,1)
        ax_cum.plot((targets_cl*count).sum(axis=0), 'r*', label = 'Target') # not scaled by numbers
        ax_cum.plot((PV_km_up*count).sum(axis=0), color = 'orange', linestyle = ':', 
                 label ='$K$-means Bounds')
        ax_cum.plot((PV_km_low*count).sum(axis=0), color = 'orange', linestyle = ':')
        ax_cum.plot(((PV_km_up+PV_km_low)*count/2).sum(axis=0), color = 'orange', linestyle = '-', 
                 label ='$K$-means Prediction')
        
        if include_ann == True:
            ax_cum.plot((ann_prediction*count).sum(axis=0), color = 'black', 
                        label = 'ANN Prediction')
            ax_cum.plot((PV_ann_up*count).sum(axis = 0), color = 'black' , linestyle = ':',
                        label = 'ANN Bounds')
            ax_cum.plot((PV_ann_low*count).sum(axis = 0), color = 'black', linestyle = ':')
        ax_cum.set_xlabel('Time, $t$', fontsize = 'large')
        ax_cum.set_ylabel('Policy Value', fontsize = 'large')
        if option_header:
            ax_cum.set_title('$K=$ '+str(n_cl), fontsize= 'large')
        if option_legend:
            ax_cum.legend() 
 
            
        
    elif option == 'statistic':
        # Compare target policy values with policy values of ANN representatives
        # For the PV of ANN representatives we take the mean of the floored (ann_PV_low)
        # and ceiled value (ann_PV_up) as our proxy
        
        ### Look at cumulative cluster fit ###
        fig_cum, ax_cum = plt.subplots(1,1)
        ax_cum.plot((targets_cl*count).sum(axis=0), 'r*', label = 'Target') # not scaled by numbers
        ax_cum.plot((PV_km_up*count).sum(axis=0), color = 'orange', linestyle = ':', 
                 label ='$K$-means Bounds')
        ax_cum.plot((PV_km_low*count).sum(axis=0), color = 'orange', linestyle = ':')
        ax_cum.plot(((PV_km_up+PV_km_low)*count/2).sum(axis=0), color = 'orange', linestyle = '-', 
                 label ='$K$-means Prediction')
        
        if include_ann == True:
            ax_cum.plot((ann_prediction*count).sum(axis=0), color = 'black', 
                        label = 'ANN Prediction')
            ax_cum.plot((PV_ann_up*count).sum(axis = 0), color = 'black' , linestyle = ':',
                        label = 'ANN Bounds')
            ax_cum.plot((PV_ann_low*count).sum(axis = 0), color = 'black', linestyle = ':')
        ax_cum.set_xlabel('Time, $t$', fontsize = 'large')
        ax_cum.set_ylabel('Policy Value', fontsize = 'large')
        if option_header:
            ax_cum.set_title('$K=$ '+str(n_cl), fontsize= 'large')
        if option_legend:
            ax_cum.legend() 
            
        
        df = pd.DataFrame(data=None, index = None, columns = ['$\overline{re}_t$', #r'$CL_{0.99,|re{}_t|}$',
                                                              '$\overline{e}_t$'])#, ' $CL_{0.99,|e{}_t|}$'])
        
        ## Cumulative
        index_cum = ((targets_cl*count).sum(axis=0)>0)
        
        # Portfolio: K-Means
        diff_km = (((PV_km_up*count+PV_km_low*count)/2).sum(axis=0)-(targets_cl*count).sum(axis=0))
        diff_km_rel = diff_km[index_cum]/(targets_cl*count).sum(axis=0)[index_cum]
        #vol_km_discr = ((PV_km_up+PV_km_low)/2*count).sum()/(targets_cl*count).sum()-1
        df.loc['$K$-means Prediction'] = (diff_km_rel.mean(),# np.percentile(np.abs(diff_km_rel),99), 
                                        diff_km.mean())#, np.percentile(np.abs(diff_km),99))
        # Portfolio: ANN prediction
        diff_pred = (ann_prediction*count).sum(axis=0)-(targets_cl*count).sum(axis=0)
        diff_pred_rel = diff_pred[index_cum]/(targets_cl*count).sum(axis=0)[index_cum]
        #vol_pred_discr = (ann_prediction*count).sum()/(targets_cl*count).sum()-1
        df.loc['ANN Prediction'] = (diff_pred_rel.mean(), #np.percentile(np.abs(diff_pred_rel),99), 
                                    diff_pred.mean())#, np.percentile(np.abs(diff_pred),99))
        
        # Portfolio: ANN via classical calculation
        diff = ((PV_ann_up*count).sum(axis=0)-(targets_cl*count).sum(axis=0))
        diff_rel = diff[index_cum]/(targets_cl*count).sum(axis=0)[index_cum]
        #vol_discr = ((PV_ann_up+PV_ann_low)/2*count).sum()/(targets_cl*count).sum()-1
        df.loc['ANN Bounds (up)'] = (diff_rel.mean(), #np.percentile(np.abs(diff_rel),99), 
                                    diff.mean())#, np.percentile(np.abs(diff),99))
        
        diff = ((PV_ann_low*count).sum(axis=0)-(targets_cl*count).sum(axis=0))
        diff_rel = diff[index_cum]/(targets_cl*count).sum(axis=0)[index_cum]
        #vol_discr = ((PV_ann_up+PV_ann_low)/2*count).sum()/(targets_cl*count).sum()-1
        df.loc['ANN Bounds (low)'] = (diff_rel.mean(), #np.percentile(np.abs(diff_rel),99), 
                                    diff.mean())#, np.percentile(np.abs(diff),99))
        
       
        return (df, targets_cl, PV_ann_low ,PV_ann_up, count)
    else:
        print('Unknown option!')
    
    return

In [11]:
## function which compares (or visualizes) agglomeration
## We want to compare the baseline kMeans with the ANN Agglomeration
## For the ANN we have to consider the predicted reserve (by part I), which
# is integrated in the ANN-Model for clustering
# and compare this to the reserve based on classical calculation and the representatives
# obtain by the ANN-Cluster-Model
# option_plot_selection: insert a a list of numbers, representing the selcted clusters to be plotted

def analyze_agglomeration_test(baseline, y, Max_min, x = None, include_ann = False, y_ann = None,
                              ann_representatives = None, ann_prediction = None, #N_modelpoints = None,
                              option = 'plot', individual_clusters = False, option_cluster_fit = True,
                              n_columns = 5, figsize = (20,30), option_plot_selection = None, 
                              ann_cluster_presort = None, option_header = True,
                              insurance_type = 'pensions', pension_age_max = 67,
                              interest_rate = 0.03,
                              ep_rate = [0.3, 0.1, 0.1, 0.1, 0.1, 0.1]):
    
    #Number of clusters
    n_cl = baseline.cluster_centers_.shape[0]
    n_cl_ann = len(ann_representatives)
    
    
    n_features = baseline.cluster_centers_.shape[1]
    # Number of members per cluster
    count = kmeans_counts(baseline.labels_,n_cl) # customized function kmeans_counts
    count_ann = kmeans_counts(ann_cluster_presort.labels_, n_cl_ann)
    # size of output (vector of policy values)
    n_out = targets.shape[1]
    N_centroids = len(ann_representatives[0]) #N_modelpoints #
    
    # Retransformed Data
    if insurance_type == 'termlife':
        data_kmeans = data_re_transform_features(baseline.cluster_centers_, Max_min, option= 'conditional')
        if include_ann == True:
            data_ann = data_re_transform_features(ann_representatives, Max_min, option= 'conditional')
        
    elif insurance_type == 'pensions':
        data_kmeans = data_re_transform_features(baseline.cluster_centers_, Max_min, option= 'standard')

        if include_ann == True:
            data_ann = {}
            for i in range(n_cl_ann):
                # transform centroids for each cluster individually
                # dtype ann_rep: dictionary
                # dtype centroids: nparrays
                data_ann[i] = [data_re_transform_features(ann_representatives[i][k], 
                                                          Max_min, option= 'standard').reshape((n_features,))
                               for k in range(N_centroids)]
    else:
        print('Error: insurance_type not known/ implemented!')
        return
   
    #return data_ann
        
    # Calculate targets for (floored and ceiled) centroids for baseline and
    # optionally: also for representatives obtained by ANN
    
#    global PV_km_low, PV_km_up, PV_ann_up, PV_ann_low
    PV_km_low = np.zeros(shape = (n_cl, n_out))
    PV_km_up = np.zeros(shape = (n_cl, n_out))
    PV_km_pred = np.zeros(shape = (n_cl, n_out))
    PV_ann_up = {} #np.zeros(shape = (n_cl, n_out))
    PV_ann_low = {} #np.zeros(shape = (n_cl, n_out))
    
    
    for i in range(n_cl):
    ##############################
        if insurance_type == 'termlife':
                
            PV_km_low[i,0:max(np.floor(data_kmeans[i,2]).astype('int')-
                                  np.floor(data_kmeans[i,2]).astype('int'),0)+1] = get_termlife_reserve_profile(
                                                        age_curr = np.floor(data_kmeans[i,0]).astype('int'), 
                                                        Sum_ins= data_kmeans[i,1],
                                                         duration = np.floor(data_kmeans[i,2]).astype('int'), 
                                                        #interest = data_kmeans[i,4],
                                                        interest = interest_rate,
                                                         age_of_contract = np.floor(data_kmeans[i,3]).astype('int'), 
                                                          option_past = False)


            PV_km_up[i,0:max(np.ceil(data_kmeans[i,2]).astype('int')-
                  np.ceil(data_kmeans[i,3]).astype('int'),0)+1] = get_termlife_reserve_profile(
                                                        age_curr = np.ceil(data_kmeans[i,0]).astype('int'), 
                                                        Sum_ins= data_kmeans[i,1],
                                                        duration = np.ceil(data_kmeans[i,2]).astype('int'), 
                                                        #interest = data_kmeans[i,4], 
                                                        interest = interest_rate,
                                                        age_of_contract = np.ceil(data_kmeans[i,3]).astype('int'), 
                                                        option_past = False)



            # Optional: Also for representatives of ANN
            # Account for that ANN might be based on less clusters but have more centroids
            if (include_ann == True) & (i<n_cl_ann):
                PV_ann_up[i] = np.zeros(shape = (N_centroids, n_out))
                PV_ann_low[i] = np.zeros(shape = (N_centroids, n_out))

                for k in range(N_centroids):
                    PV_ann_low[i][k][0:max(np.floor(data_ann[i,2]).astype('int')-
                                  np.floor(data_ann[i,3]).astype('int'),0)+1] = get_termlife_reserve_profile(
                                                                age_curr = np.floor(data_ann[i][k,0]).astype('int'), 
                                                                Sum_ins= data_ann[i][k,1],
                                                                duration = np.floor(data_ann[i][k,2]).astype('int'), 
                                                                #interest = data_ann[i,4], 
                                                                interest = interest_rate,
                                                                age_of_contract = np.floor(data_ann[i][k,3]).astype('int'), 
                                                                option_past = False)

                    PV_ann_up[i][k][0:max(np.ceil(data_ann[i,2]).astype('int')-
                                  np.ceil(data_ann[i,3]).astype('int'),0)+1] = get_termlife_reserve_profile(
                                                                age_curr = np.ceil(data_ann[i][k,0]).astype('int'), 
                                                                Sum_ins= data_ann[i][k,1],
                                                                duration = np.ceil(data_ann[i][k,2]).astype('int'), 
                                                                #interest = data_ann[i,4], 
                                                                interest = interest_rate,
                                                                age_of_contract = np.ceil(data_ann[i][k,3]).astype('int'), 
                                                                option_past = False)


    ############################
        else: # insurance_type == 'pensions':
            # input_used = ['Fund','Age', 'Salary', 'Salary_scale', 'Contribution', 'interest_rate']
            # Account for that ANN might be based on less clusters but have more centroids
            
                #for k in range(N_centroids):
            PV_km_low[i,0:max(pension_age_max- np.floor(data_kmeans[i,1]).astype('int'),0)+1] =  get_pension_reserve(fund_accum = data_kmeans[i,0], 
                                     age = np.floor(data_kmeans[i,1]).astype('int'), 
                                     salary = data_kmeans[i,2], salary_scale = data_kmeans[i,3], 
                                     contribution = data_kmeans[i,4], 
                                     A = 0.00022, B = 2.7*10**(-6), c = 1.124, 
                                    #interest = data_kmeans[i,5],
                                     interest = interest_rate,
                                     pension_age_max = pension_age_max, 
                                     early_pension = ep_rate)

            PV_km_up[i,0:max(pension_age_max- np.ceil(data_kmeans[i,1]).astype('int'),0)+1] = get_pension_reserve(fund_accum =data_kmeans[i,0], 
                                     age = np.ceil(data_kmeans[i,1]).astype('int'), 
                                     salary = data_kmeans[i,2], salary_scale = data_kmeans[i,3], 
                                     contribution = data_kmeans[i,4], 
                                     A = 0.00022, B = 2.7*10**(-6), c = 1.124, 
                                      #interest = data_kmeans[i,5],
                                      interest = interest_rate,
                                     pension_age_max = pension_age_max, 
                                     early_pension = ep_rate)

                # Optional: Also for representatives of ANN
            if (include_ann == True)& (i<n_cl_ann):
                PV_ann_up[i] = np.zeros(shape = (N_centroids, n_out))
                PV_ann_low[i] = np.zeros(shape = (N_centroids, n_out))

                for k in range(N_centroids):
                    PV_ann_low[i][k][0:max(pension_age_max- np.ceil(data_ann[i][k][1]).astype('int'),0)+1] = get_pension_reserve(
                                             fund_accum =data_ann[i][k][0], 
                                             age = np.ceil(data_ann[i][k][1]).astype('int'), 
                                             salary = data_ann[i][k][2], salary_scale = data_ann[i][k][3], 
                                             contribution = data_ann[i][k][4], 
                                             A = 0.00022, B = 2.7*10**(-6), c = 1.124, 
                                              #interest = data_ann[i,5],
                                              interest = interest_rate,
                                             pension_age_max = pension_age_max, 
                                             early_pension = ep_rate)

                    PV_ann_up[i][k][0:max(pension_age_max- np.floor(data_ann[i][k][1]).astype('int'),0)+1] = get_pension_reserve(
                                             fund_accum =data_ann[i][k][0], 
                                             age = np.floor(data_ann[i][k][1]).astype('int'), 
                                             salary = data_ann[i][k][2], salary_scale = data_ann[i][k][3], 
                                             contribution = data_ann[i][k][4], 
                                             A = 0.00022, B = 2.7*10**(-6), c = 1.124, 
                                            #interest = data_ann[i,5],
                                            interest = interest_rate,
                                             pension_age_max = pension_age_max, 
                                             early_pension = ep_rate)



    ##############################################
    # Calculate actual targets per cluster 
    targets_cl = np.zeros(shape = (n_cl, n_output))
    for i in range(n_cl):
        index = (baseline.labels_ == i)
        targets_cl[i,:] = targets[index,:].sum(axis=0)/count[i]

    PV_km_pred = (PV_km_up[:,:]+PV_km_low[:,:])/2
    y_pred_up = sum(PV_ann_up[i][k]*count_ann[i]/N_centroids for i in range(n_cl_ann) for k in range(N_centroids))
    y_pred_low = sum(PV_ann_low[i][k]*count_ann[i]/N_centroids for i in range(n_cl_ann) for k in range(N_centroids))


    
    if option == 'plot':
        
        if (individual_clusters == True) & ((n_cl==n_cl_ann)or (include_ann == False)):
            
            
            if option_plot_selection == None: # Plot all clusters C_1,..,C_K
                fig, ax = plt.subplots(nrows = np.ceil(n_cl/n_columns).astype('int'), 
                                       ncols = n_columns, figsize = figsize)
                if n_cl > 1:
                    ax = ax.flatten()
                else:
                    ax = [ax]

                for i in range(n_cl):
                    # Actual Targets
                    ax[i].plot(targets_cl[i,:], 'r*', label = 'Target')
                    # Reserve based on K-Means clustering
                    
                    ax[i].plot(PV_km_pred[i,:], linestyle = '-', color = 'orange', label = 'K-Means Prediction')
                    ax[i].plot(PV_km_up[i,:], linestyle = ':', color = 'orange', #linewidth = 5, # marker = 'o', 
                               label = 'K-Means Bounds')
                    ax[i].plot(PV_km_low[i,:], linestyle = ':', color = 'orange')#, linewidth = 5)#,marker = 'o')

                    if i%n_columns==0: # first column
                        ax[i].set_ylabel('Policy Value', fontsize = 'large')
                    if i>= (n_columns*(np.ceil(n_cl/n_columns).astype('int')-1)): # last row
                        ax[i].set_xlabel('Time, $t$', fontsize = 'large')

                    if (include_ann == True) & (i<n_cl_ann):
                        
                        # Predicted Reserve by ANN (overall)
                        ax[i].plot(sum(ann_prediction[i][k] for k in range(N_centroids))/N_centroids,
                                       linestyle = '-', color = 'black', 
                                       label = 'ANN Prediction')
                        ax[i].plot(sum(PV_ann_up[i][k] for k in range(N_centroids))/N_centroids,
                                       linestyle = '--', color = 'black', 
                                       label = 'ANN Bounds')
                        ax[i].plot(sum(PV_ann_low[i][k] for k in range(N_centroids))/N_centroids,
                                       linestyle = '--', color = 'black')
                        if N_centroids >1:
                            for k in range(N_centroids):

                                # Predicted Reserve by ANN (per centroid)
                                if k == 0:
                                    ax[i].plot(ann_prediction[i][k], linestyle = '-', color = 'grey', 
                                               label = 'ANN Prediction (MP)')
                                else:
                                    ax[i].plot(ann_prediction[i][k], linestyle = '-', color = 'grey')


                    if i == 0:
                        ax[i].legend()

                fig.tight_layout()
            else: ### Plot only selection of clusters C_1,...,C_[option_plot_selection]
                
                
                fig, ax = plt.subplots(nrows = np.ceil(len(option_plot_selection)/n_columns).astype('int'), 
                                       ncols = n_columns, figsize = figsize)
                ax = ax.flatten()

                for i in range(len(option_plot_selection)):
                    # Actual Targets
                    ax[i].plot(targets_cl[option_plot_selection[i],:], 'r*', label = 'Target')
                    # Reserve based on K-Means clustering
                    ax[i].plot(PV_km_pred[option_plot_selection[i],:], linestyle = '-', color = 'orange', 
                               label = 'K-Means Prediction')
                    ax[i].plot(PV_km_up[option_plot_selection[i],:], linestyle = ':', color = 'orange', 
                               label = 'K-Means Bounds')
                    ax[i].plot(PV_km_low[option_plot_selection[i],:], linestyle = ':', color = 'orange')

                    if i%n_columns==0: # first column
                        ax[i].set_ylabel('Policy Value', fontsize = 'large')
                    if i>= (len(option_plot_selection)-n_columns): # last row
                        ax[i].set_xlabel('Time, $t$', fontsize = 'large')

                    if (include_ann == True) & (i<n_cl_ann):
                        
                        # Predicted Reserve by ANN (overall)
                        ax[i].plot(sum(ann_prediction[option_plot_selection[i]][k] for k in range(N_centroids))/N_centroids,
                                       linestyle = '-', color = 'black', label = 'ANN Prediction')
                        ax[i].plot(sum(PV_ann_up[option_plot_selection[i]][k] for k in range(N_centroids))/N_centroids,
                                       linestyle = ':', color = 'black',label = 'ANN Bounds')
                        ax[i].plot(sum(PV_ann_low[option_plot_selection[i]][k] for k in range(N_centroids))/N_centroids,
                                       linestyle = ':', color = 'black')
                        
                        if N_centroids>1:
                            for k in range(N_centroids):
                            # Predicted Reserve by ANN (per centroid)
                                if k == 0:
                                    ax[i].plot(ann_prediction[option_plot_selection[i]][k], linestyle = '-', color = 'grey', 
                                               label = 'ANN Prediction (MP)')
                                else:
                                    ax[i].plot(ann_prediction[option_plot_selection[i]][k], linestyle = '-', color = 'grey')

                    if i == 0:
                        ax[i].legend()
                plt.tight_layout()
            
        ############################################################################
        ### Look at cumulative cluster fit ###
        fig_cum, ax_cum = plt.subplots(1,1)

        ax_cum.plot(targets.sum(axis=0), 'r*', label = 'Target') # not scaled by numbers
        ax_cum.plot((PV_km_pred*count).sum(axis=0), color = 'orange', linestyle = '-', 
                 label ='K-Means Prediction')
        ax_cum.plot((PV_km_up*count).sum(axis=0), color = 'orange', linestyle = ':', 
                 label ='K-Means Bounds')
        ax_cum.plot((PV_km_low*count).sum(axis=0), color = 'orange', linestyle = ':')
        if include_ann == True:
            ax_cum.plot(sum(ann_prediction[i][k]*count_ann[i]/N_centroids for i in range(n_cl_ann) for k in range(N_centroids)), 
                        color = 'black', label = 'ANN Prediction')
            
            ax_cum.plot(sum(PV_ann_up[i][k]*count_ann[i]/N_centroids for i in range(n_cl_ann) for k in range(N_centroids)), 
                        color = 'black', linestyle = ':', label = 'ANN Bounds')
            ax_cum.plot(sum(PV_ann_low[i][k]*count_ann[i]/N_centroids for i in range(n_cl_ann) for k in range(N_centroids)), 
                        color = 'black', linestyle = ':')
        ax_cum.set_xlabel('Time, $t$', fontsize = 'large')
        ax_cum.set_ylabel('Policy Value', fontsize = 'large')
        if option_header:
            ax_cum.set_title(r'K-Means: $K= {}$, ANN: $K= {}$, $C= {}$ '.format(n_cl, n_cl_ann, N_centroids), fontsize= 'large')
        ax_cum.legend() 
        
        #return PV_ann_up, PV_ann_low
            
#######################################################################################        
    elif option == 'statistic':
        
        # Compare target policy values with policy values of ANN representatives
        # For the PV of ANN representatives we take the mean of the floored (ann_PV_low)
        # and ceiled value (ann_PV_up) as our proxy
        
        #df = pd.DataFrame(data=None, index = None, columns = ['min re${}_t$','mean re${}_t$',
         #                                                     'max re${}_t$',r'$D$'])
        
        df = pd.DataFrame(data=None, index = None, columns = ['$\overline{re}_t$', #r'$CL_{0.99,|re{}_t|}$',
                                                              '$\overline{e}_t$'])#, ' $CL_{0.99,|e{}_t|}$'])
 ################################################################
        ### Look at cumulative cluster fit ###
        fig_cum, ax_cum = plt.subplots(1,1)

        ax_cum.plot(targets.sum(axis=0), 'r*', label = 'Target') # not scaled by numbers
        ax_cum.plot((PV_km_pred*count).sum(axis=0), color = 'orange', linestyle = '-', 
                 label ='K-Means Prediction')
        ax_cum.plot((PV_km_up*count).sum(axis=0), color = 'orange', linestyle = ':', 
                 label ='K-Means Bounds')
        ax_cum.plot((PV_km_low*count).sum(axis=0), color = 'orange', linestyle = ':')
        if include_ann == True:
            ax_cum.plot(sum(ann_prediction[i][k]*count_ann[i]/N_centroids for i in range(n_cl_ann) for k in range(N_centroids)), 
                        color = 'black', label = 'ANN Prediction')
            
            ax_cum.plot(sum(PV_ann_up[i][k]*count_ann[i]/N_centroids for i in range(n_cl_ann) for k in range(N_centroids)), 
                        color = 'black', linestyle = ':', label = 'ANN Bounds')
            ax_cum.plot(sum(PV_ann_low[i][k]*count_ann[i]/N_centroids for i in range(n_cl_ann) for k in range(N_centroids)), 
                        color = 'black', linestyle = ':')
        ax_cum.set_xlabel('Time, $t$', fontsize = 'large')
        ax_cum.set_ylabel('Policy Value', fontsize = 'large')
        if option_header:
            ax_cum.set_title(r'K-Means: $K= {}$, ANN: $K= {}$, $C= {}$ '.format(n_cl, n_cl_ann, N_centroids), fontsize= 'large')
        ax_cum.legend() 
   ###############################################################    
        
        ## Cumulative
        index_cum = (targets.sum(axis=0)>0)
        
        # Portfolio: K-Means
        diff_km = (PV_km_pred*count).sum(axis=0) - targets.sum(axis=0)
        diff_km_rel = diff_km[index_cum]/targets.sum(axis=0)[index_cum]
        #vol_km_discr = ((PV_km_up+PV_km_low)/2*count).sum()/(targets_cl*count).sum()-1
        df.loc['$K$-means Prediction'] = (diff_km_rel.mean(), #np.percentile(np.abs(diff_km_rel),99), 
                               diff_km.mean())#, np.percentile(np.abs(diff_km),99))
        
        # Portfolio: ANN via classical calculation
        #diff = (((PV_ann_up*count+PV_ann_low*count)/2).sum(axis=0)-(targets_cl*count).sum(axis=0))
        diff_up = sum((PV_ann_up[i][k])*count_ann[i]/N_centroids for i in range(n_cl_ann) for k in range(N_centroids)) -targets.sum(axis=0)
        diff_rel_up = diff_up[index_cum]/targets.sum(axis=0)[index_cum]
        diff_low = sum((PV_ann_low[i][k])*count_ann[i]/N_centroids for i in range(n_cl_ann) for k in range(N_centroids)) -targets.sum(axis=0)
        diff_rel_low = diff_low[index_cum]/(targets_cl*count).sum(axis=0)[index_cum]
        
        
        
        
        # Portfolio: ANN prediction
        diff_pred = sum(ann_prediction[i][k]*count_ann[i]/N_centroids for i in range(n_cl_ann) for k in range(N_centroids))-targets.sum(axis=0)
        diff_pred_rel = diff_pred[index_cum]/targets.sum(axis=0)[index_cum]#(targets_cl*count).sum(axis=0)[index_cum]
        #vol_pred_discr = (ann_prediction*count).sum()/(targets_cl*count).sum()-1
        df.loc['ANN Prediction'] = (diff_pred_rel.mean(),# np.percentile(np.abs(diff_pred_rel),99), 
                                    diff_pred.mean())#, np.percentile(np.abs(diff_pred),99)) 
        
        #diff = sum(PV_ann_up[i][k]*count_ann[i] for i in range(n_cl_ann) for k in range(N_centroids))/N_centroids -(targets_cl*count).sum(axis=0)
        #diff_rel = diff[index_cum]/(targets_cl*count).sum(axis=0)[index_cum]
        #vol_discr = ((PV_ann_up+PV_ann_low)/2*count).sum()/(targets_cl*count).sum()-1
        df.loc['ANN Bound (up)'] = (diff_rel_up.mean(),# np.percentile(np.abs(diff_rel),99), 
                                    diff_up.mean())#, np.percentile(np.abs(diff),99))
        
        #diff = sum(PV_ann_low[i][k]*count_ann[i] for i in range(n_cl_ann) for k in range(N_centroids))/N_centroids -(targets_cl*count).sum(axis=0)
        #diff_rel = diff[index_cum]/(targets_cl*count).sum(axis=0)[index_cum]
        #vol_discr = ((PV_ann_up+PV_ann_low)/2*count).sum()/(targets_cl*count).sum()-1
        df.loc['ANN Bound (low)'] = (diff_rel_low.mean(),# np.percentile(np.abs(diff_rel),99), 
                                    diff_low.mean())#, np.percentile(np.abs(diff),99))  
        
        
        ann_pred = sum(ann_prediction[i][k]*count_ann[i]/N_centroids for i in range(n_cl_ann) for k in range(N_centroids))
        ann_pred_up = sum(PV_ann_up[i][k]*count_ann[i]/N_centroids for i in range(n_cl_ann) for k in range(N_centroids))
        ann_pred_low = sum(PV_ann_low[i][k]*count_ann[i]/N_centroids for i in range(n_cl_ann) for k in range(N_centroids))
        return df, [diff_km, diff_km_rel], [diff_pred, diff_pred_rel], [diff_up, diff_rel_up], [diff_low, diff_rel_low],  [ann_pred_up, ann_pred_low, ann_pred], [PV_ann_up, PV_ann_low]
    else:
        print('Unknown option!')
    return